In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import time
import sys

from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

In [2]:
DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")
RESULT_DIR = Path("../reports/results")

PREDICTION_DIR = RESULT_DIR / "predictions"

PREDICTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Directories configured successfully!")

Directories configured successfully!


In [3]:
model = joblib.load(
    MODEL_DIR / "best_model_pipeline.pkl"
)

print("Final model loaded successfully!")

print("\nSelected Classifier:")

print(model.named_steps["classifier"])

Final model loaded successfully!

Selected Classifier:
RandomForestClassifier(n_jobs=-1, random_state=42)


In [4]:
with open(
    RESULT_DIR / "final_model_metadata.json",
    "r"
) as file:

    metadata = json.load(file)

feature_names = metadata["feature_names"]

numerical_features = metadata["numerical_features"]

categorical_features = metadata["categorical_features"]

print("Selected Model:", metadata["model_name"])

print("Model Type:", metadata["model_type"])

print("Total Features:", len(feature_names))

print("Numerical Features:", len(numerical_features))

print("Categorical Features:", len(categorical_features))

print("Test Accuracy:",
      round(metadata["accuracy"] * 100, 2), "%")

Selected Model: Random Forest
Model Type: Baseline
Total Features: 42
Numerical Features: 39
Categorical Features: 3
Test Accuracy: 87.22 %


In [5]:
feature_df = pd.DataFrame({

    "Feature Name": feature_names,

    "Data Type": [
        "Categorical"
        if feature in categorical_features
        else "Numerical"
        for feature in feature_names
    ]
})

display(feature_df)

,Feature Name,Data Type
0,dur,Numerical
1,proto,Categorical
2,service,Categorical
3,state,Categorical
4,spkts,Numerical
5,dpkts,Numerical
6,sbytes,Numerical
7,dbytes,Numerical
8,rate,Numerical
9,sttl,Numerical


In [6]:
X_test = pd.read_csv(
    DATA_DIR / "X_test.csv"
)

y_test = pd.read_csv(
    DATA_DIR / "y_test.csv"
)["label"]

print("Available Sample Records:", len(X_test))

display(X_test.head())

Available Sample Records: 82332


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports
0,0.000011,udp,-,INT,2,0,496,0,90909.0902,254,...,1,1,1,2,0,0,0,1,2,0
1,0.000008,udp,-,INT,2,0,1762,0,125000.0003,254,...,1,1,1,2,0,0,0,1,2,0
2,0.000005,udp,-,INT,2,0,1068,0,200000.0051,254,...,1,1,1,3,0,0,0,1,3,0
3,0.000006,udp,-,INT,2,0,900,0,166666.6608,254,...,2,2,1,3,0,0,0,2,3,0
4,0.000010,udp,-,INT,2,0,2126,0,100000.0025,254,...,2,2,1,3,0,0,0,2,3,0


In [7]:
def validate_input(data):

    if not isinstance(data, pd.DataFrame):
        raise TypeError("Input must be a DataFrame.")

    if data.empty:
        raise ValueError("Input data is empty.")

    if data.columns.duplicated().any():
        raise ValueError("Duplicate column names found.")

    data = data.copy()

    missing_features = [
        feature for feature in feature_names
        if feature not in data.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing features: {missing_features}"
        )

    data = data[feature_names]

    for feature in numerical_features:

        data[feature] = pd.to_numeric(
            data[feature],
            errors="raise"
        )

    for feature in categorical_features:

        data[feature] = data[feature].map(
            lambda value: str(value).strip()
            if pd.notna(value)
            else np.nan
        )

    data = data.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if data.isna().all(axis=1).any():
        raise ValueError("Completely empty record found.")

    return data

In [8]:
def predict_intrusion(data):

    validated_data = validate_input(data)

    predictions = model.predict(
        validated_data
    )

    probabilities = model.predict_proba(
        validated_data
    )

    malicious_index = list(model.classes_).index(1)

    malicious_probabilities = probabilities[
        :, malicious_index
    ]

    results = pd.DataFrame({

        "Predicted Label": predictions,

        "Prediction": [
            "Malicious" if label == 1
            else "Normal"
            for label in predictions
        ],

        "Malicious Probability":
            malicious_probabilities,

        "Normal Probability":
            1 - malicious_probabilities
    })

    return results

In [9]:
normal_index = y_test[
    y_test == 0
].index[0]

normal_sample = X_test.loc[
    [normal_index]
]

normal_result = predict_intrusion(
    normal_sample
)

print("Actual Class: Normal")

display(normal_result)

Actual Class: Normal


,Predicted Label,Prediction,Malicious Probability,Normal Probability
0,1,Malicious,0.754286,0.245714


In [10]:
malicious_index = y_test[
    y_test == 1
].index[0]

malicious_sample = X_test.loc[
    [malicious_index]
]

malicious_result = predict_intrusion(
    malicious_sample
)

print("Actual Class: Malicious")

display(malicious_result)

Actual Class: Malicious


,Predicted Label,Prediction,Malicious Probability,Normal Probability
0,1,Malicious,0.99,0.01


In [11]:
normal_indices = y_test[
    y_test == 0
].sample(
    5,
    random_state=42
).index

malicious_indices = y_test[
    y_test == 1
].sample(
    5,
    random_state=42
).index

sample_indices = (
    normal_indices.tolist() +
    malicious_indices.tolist()
)

sample_data = X_test.loc[
    sample_indices
]

batch_results = predict_intrusion(
    sample_data
)

batch_results["Actual Label"] = y_test.loc[
    sample_indices
].values

batch_results["Correct Prediction"] = (
    batch_results["Predicted Label"] ==
    batch_results["Actual Label"]
)

display(batch_results.round(4))

,Predicted Label,Prediction,Malicious Probability,Normal Probability,Actual Label,Correct Prediction
0,0,Normal,0.33,0.67,0,True
1,0,Normal,0.02,0.98,0,True
2,1,Malicious,0.91,0.09,0,False
3,1,Malicious,0.80,0.20,0,False
4,0,Normal,0.00,1.00,0,True
5,1,Malicious,1.00,0.00,1,True
6,1,Malicious,0.98,0.02,1,True
7,1,Malicious,0.79,0.21,1,True
8,1,Malicious,1.00,0.00,1,True
9,1,Malicious,1.00,0.00,1,True


In [12]:
def get_risk_level(probability):

    if probability < 0.20:
        return "Very Low"

    elif probability < 0.40:
        return "Low"

    elif probability < 0.60:
        return "Moderate"

    elif probability < 0.80:
        return "High"

    else:
        return "Critical"

In [13]:
batch_results["Risk Level"] = (
    batch_results["Malicious Probability"]
    .apply(get_risk_level)
)

display(batch_results.round(4))

,Predicted Label,Prediction,Malicious Probability,Normal Probability,Actual Label,Correct Prediction,Risk Level
0,0,Normal,0.33,0.67,0,True,Low
1,0,Normal,0.02,0.98,0,True,Very Low
2,1,Malicious,0.91,0.09,0,False,Critical
3,1,Malicious,0.80,0.20,0,False,Critical
4,0,Normal,0.00,1.00,0,True,Very Low
5,1,Malicious,1.00,0.00,1,True,Critical
6,1,Malicious,0.98,0.02,1,True,Critical
7,1,Malicious,0.79,0.21,1,True,High
8,1,Malicious,1.00,0.00,1,True,Critical
9,1,Malicious,1.00,0.00,1,True,Critical


In [14]:
input_template = pd.DataFrame(
    columns=feature_names
)

input_template.to_csv(
    PREDICTION_DIR / "network_input_template.csv",
    index=False
)

print("Input template created successfully!")

display(input_template)

Input template created successfully!


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports


In [15]:
custom_input = normal_sample.copy()

custom_input.loc[:, "dur"] = 0.5

custom_input.loc[:, "proto"] = "tcp"

custom_input.loc[:, "service"] = "http"

custom_input.loc[:, "spkts"] = 20

custom_input.loc[:, "dpkts"] = 10

custom_input.loc[:, "sbytes"] = 1500

custom_input.loc[:, "dbytes"] = 800

display(custom_input)

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports
0,0.5,tcp,http,INT,20,10,1500,800,90909.0902,254,...,1,1,1,2,0,0,0,1,2,0


In [16]:
custom_result = predict_intrusion(
    custom_input
)

custom_result["Risk Level"] = (
    custom_result["Malicious Probability"]
    .apply(get_risk_level)
)

display(custom_result.round(4))

,Predicted Label,Prediction,Malicious Probability,Normal Probability,Risk Level
0,1,Malicious,0.77,0.23,High


In [17]:
normal_sample.to_csv(
    PREDICTION_DIR / "normal_sample.csv",
    index=False
)

malicious_sample.to_csv(
    PREDICTION_DIR / "malicious_sample.csv",
    index=False
)

sample_data.to_csv(
    PREDICTION_DIR / "batch_sample.csv",
    index=False
)

custom_input.to_csv(
    PREDICTION_DIR / "custom_sample.csv",
    index=False
)

print("All sample CSV files generated!")

All sample CSV files generated!


In [18]:
uploaded_data = pd.read_csv(
    PREDICTION_DIR / "batch_sample.csv"
)

csv_results = predict_intrusion(
    uploaded_data
)

csv_results["Risk Level"] = (
    csv_results["Malicious Probability"]
    .apply(get_risk_level)
)

display(csv_results.round(4))

,Predicted Label,Prediction,Malicious Probability,Normal Probability,Risk Level
0,0,Normal,0.33,0.67,Low
1,0,Normal,0.02,0.98,Very Low
2,1,Malicious,0.91,0.09,Critical
3,1,Malicious,0.80,0.20,Critical
4,0,Normal,0.00,1.00,Very Low
5,1,Malicious,1.00,0.00,Critical
6,1,Malicious,0.98,0.02,Critical
7,1,Malicious,0.79,0.21,High
8,1,Malicious,1.00,0.00,Critical
9,1,Malicious,1.00,0.00,Critical


In [19]:
total_records = len(csv_results)

normal_count = (
    csv_results["Predicted Label"] == 0
).sum()

malicious_count = (
    csv_results["Predicted Label"] == 1
).sum()

print("Total Records:", total_records)

print("Normal Records:", normal_count)

print("Malicious Records:", malicious_count)

print(
    "Predicted Malicious Traffic Percentage:",
    round(malicious_count / total_records * 100, 2),
    "%"
)

print("\nRisk Distribution:")

print(csv_results["Risk Level"].value_counts())

Total Records: 10
Normal Records: 3
Malicious Records: 7
Predicted Malicious Traffic Percentage: 70.0 %

Risk Distribution:
Risk Level
Critical    6
Very Low    2
Low         1
High        1
Name: count, dtype: int64


In [20]:
final_prediction_output = pd.concat([

    uploaded_data.reset_index(drop=True),

    csv_results.reset_index(drop=True)

], axis=1)

final_prediction_output.to_csv(
    PREDICTION_DIR / "batch_prediction_results.csv",
    index=False
)

custom_result.to_csv(
    PREDICTION_DIR / "custom_prediction_result.csv",
    index=False
)

print("Prediction results saved successfully!")

Prediction results saved successfully!


In [21]:
from src.predict import IntrusionPredictor

predictor = IntrusionPredictor()

print("Prediction module loaded successfully!")

Prediction module loaded successfully!


In [22]:
result = predictor.predict(
    normal_sample
)

display(result)

,Predicted Label,Prediction,Malicious Probability,Normal Probability,Risk Level
0,1,Malicious,0.754286,0.245714,High


In [23]:
result = predictor.predict_csv(
    PREDICTION_DIR / "batch_sample.csv"
)

display(result)

,Predicted Label,Prediction,Malicious Probability,Normal Probability,Risk Level
0,0,Normal,0.33,0.67,Low
1,0,Normal,0.02,0.98,Very Low
2,1,Malicious,0.91,0.09,Critical
3,1,Malicious,0.80,0.20,Critical
4,0,Normal,0.00,1.00,Very Low
5,1,Malicious,1.00,0.00,Critical
6,1,Malicious,0.98,0.02,Critical
7,1,Malicious,0.79,0.21,High
8,1,Malicious,1.00,0.00,Critical
9,1,Malicious,1.00,0.00,Critical


In [24]:
invalid_input = pd.DataFrame({
    "dur": [0.5],
    "proto": ["tcp"]
})

try:

    predictor.predict(invalid_input)

except ValueError as error:

    print("Validation Error:", error)

Validation Error: Missing features: ['service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']


In [25]:
notebook_result = predict_intrusion(
    sample_data
)

module_result = predictor.predict(
    sample_data
)

assert np.array_equal(

    notebook_result["Predicted Label"].values,

    module_result["Predicted Label"].values

)

assert np.allclose(

    notebook_result["Malicious Probability"].values,

    module_result["Malicious Probability"].values

)

print("Prediction consistency verified!")

print("Prediction module is ready for deployment!")

Prediction consistency verified!
Prediction module is ready for deployment!
